# 02 — Exploratory Data Analysis

Descriptive statistics and visualisations on the **full cleaned datasets** from `data/processed/`.
Nothing expensive (no embeddings, no BERTopic, no sampling) — just fast counts and plots.

**Covers**
1. Rating distribution per app
2. Review length (character + word count) distribution per app
3. Developer response rate comparison
4. Review volume over time per app
5. Top 15 unigrams and bigrams by raw frequency per app

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from IPython.display import display

# ── Style ────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
APP_COLORS = {"healthifyme": "#2196F3", "myfitnesspal": "#FF9800"}
FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

print("Libraries loaded.")

In [ ]:
hfy = pd.read_csv("../data/processed/healthifyme_cleaned.csv",  low_memory=False,
                  parse_dates=["review_date"])
mfp = pd.read_csv("../data/processed/myfitnesspal_cleaned.csv", low_memory=False,
                  parse_dates=["review_date"])
combined = pd.concat([hfy, mfp], ignore_index=True)

print(f"HealthifyMe : {len(hfy):>8,} rows  |  {hfy['review_date'].min().date()} → {hfy['review_date'].max().date()}")
print(f"MyFitnessPal: {len(mfp):>8,} rows  |  {mfp['review_date'].min().date()} → {mfp['review_date'].max().date()}")

## 1  Rating Distribution

Both datasets are heavily right-skewed (the classic J-curve of app store reviews). Looking at the
absolute counts hides the skew — the normalised % view shows HealthifyMe's comparatively larger
1-star share, which is the more analytically interesting signal.

In [ ]:
# ── Count table ──────────────────────────────────────────────────────────────
rating_counts = (
    combined.groupby(["app_label", "rating"])
    .size()
    .rename("count")
    .reset_index()
)
rating_pct = rating_counts.copy()
totals = rating_pct.groupby("app_label")["count"].transform("sum")
rating_pct["pct"] = rating_pct["count"] / totals * 100

pivot_pct = rating_pct.pivot(index="rating", columns="app_label", values="pct").round(1)
pivot_cnt = rating_counts.pivot(index="rating", columns="app_label", values="count")

print("Counts:")
display(pivot_cnt)
print("\n% of each app's reviews:")
display(pivot_pct)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, (app, grp) in zip(axes, rating_pct.groupby("app_label")):
    bars = ax.bar(
        grp["rating"].astype(str),
        grp["pct"],
        color=APP_COLORS[app],
        edgecolor="white",
        linewidth=0.8,
    )
    for bar, (_, row) in zip(bars, grp.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{row['pct']:.1f}%",
            ha="center", va="bottom", fontsize=9,
        )
    ax.set_title(app.replace("myfitnesspal", "MyFitnessPal").replace("healthifyme", "HealthifyMe"),
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Star Rating")
    ax.set_ylabel("% of reviews")
    ax.set_ylim(0, grp["pct"].max() * 1.18)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

fig.suptitle("Rating Distribution (% of each app's reviews)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_rating_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 2  Review Length Distribution

Review length shapes what BERTopic and the sentiment models will see.  
Very short reviews (< 20 chars) are often uninformative ("great app", "👍").  
Very long reviews (> 500 chars) often carry the most nuanced sentiment — and the mixed-sentiment
patterns we flagged tend to live here.

In [ ]:
for df, name in [(hfy, "healthifyme"), (mfp, "myfitnesspal")]:
    df["char_len"] = df["review_description"].str.len()
    df["word_len"] = df["review_description"].str.split().str.len()

# Summary stats
len_stats = (
    combined.assign(
        char_len=combined["review_description"].str.len(),
        word_len=combined["review_description"].str.split().str.len(),
    )
    .groupby("app_label")[["char_len", "word_len"]]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])
    .round(1)
)
display(len_stats)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for row_i, (metric, label, cap) in enumerate([
    ("char_len", "Characters", 1500),
    ("word_len", "Words",       250),
]):
    for col_i, (app, df) in enumerate([("healthifyme", hfy), ("myfitnesspal", mfp)]):
        ax = axes[row_i][col_i]
        data = df[metric].clip(upper=cap)
        ax.hist(data, bins=80, color=APP_COLORS[app], edgecolor="none", alpha=0.85)

        med = df[metric].median()
        p90 = df[metric].quantile(0.90)
        ax.axvline(med, color="black",  linestyle="--", linewidth=1.2, label=f"Median={med:.0f}")
        ax.axvline(p90, color="crimson", linestyle=":",  linewidth=1.2, label=f"P90={p90:.0f}")

        title = app.replace("myfitnesspal", "MyFitnessPal").replace("healthifyme", "HealthifyMe")
        ax.set_title(f"{title} — {label}", fontsize=11, fontweight="bold")
        ax.set_xlabel(f"{label} (capped at {cap:,})")
        ax.set_ylabel("Reviews")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        ax.legend(fontsize=9)

fig.suptitle("Review Length Distribution", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_review_length.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Breakdown of very short vs moderate vs long reviews
bins   = [0, 20, 100, 300, 9999]
labels = ["<20 chars (noise)", "20–100 (short)", "100–300 (medium)", "300+ (long)"]

for df, name in [(hfy, "HealthifyMe"), (mfp, "MyFitnessPal")]:
    df["len_bucket"] = pd.cut(df["char_len"], bins=bins, labels=labels, right=False)
    tbl = df["len_bucket"].value_counts().sort_index().rename("count").to_frame()
    tbl["pct"] = (tbl["count"] / len(df) * 100).round(1)
    print(f"\n{name}:")
    display(tbl)

## 3  Developer Response Rate

HealthifyMe responds to **58.5%** of reviews; MyFitnessPal to only **10.6%**.
This is a large operational difference. HFY's high response rate may suppress some 1-star ratings
(users update reviews after support contact), which could affect sentiment comparisons.

In [ ]:
resp_df = (
    combined.groupby("app_label")["has_dev_response"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("pct")
    .reset_index()
)
resp_counts = (
    combined.groupby(["app_label", "has_dev_response"])
    .size()
    .rename("count")
    .reset_index()
)

responded = resp_df[resp_df["has_dev_response"] == True].set_index("app_label")["pct"]
print("Response rate (% of reviews with a developer reply):")
print(responded.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: grouped bar — responded vs not ─────────────────────────────────────
ax = axes[0]
apps    = ["healthifyme", "myfitnesspal"]
labels  = ["HealthifyMe", "MyFitnessPal"]
yes_pct = [responded.get(a, 0) for a in apps]
no_pct  = [100 - p for p in yes_pct]
x = np.arange(len(apps))
w = 0.35

b1 = ax.bar(x - w/2, yes_pct, w, label="Has dev response",    color=[APP_COLORS[a] for a in apps], alpha=0.9)
b2 = ax.bar(x + w/2, no_pct,  w, label="No response",          color="#BDBDBD", alpha=0.7)
for bar, val in zip(b1, yes_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("% of reviews")
ax.set_title("Developer Response Rate", fontsize=12, fontweight="bold")
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.legend(fontsize=9)

# ── Right: response rate broken down by star rating ───────────────────────────
ax2 = axes[1]
resp_by_rating = (
    combined.groupby(["app_label", "rating"])["has_dev_response"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"has_dev_response": "resp_pct"})
)
for app in apps:
    sub = resp_by_rating[resp_by_rating["app_label"] == app]
    ax2.plot(sub["rating"], sub["resp_pct"],
             marker="o", linewidth=2, markersize=6,
             color=APP_COLORS[app],
             label=app.replace("myfitnesspal", "MyFitnessPal").replace("healthifyme", "HealthifyMe"))

ax2.set_xlabel("Star Rating")
ax2.set_ylabel("% reviews with dev response")
ax2.set_title("Response Rate by Star Rating", fontsize=12, fontweight="bold")
ax2.set_xticks([1, 2, 3, 4, 5])
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_dev_response_rate.png", dpi=150, bbox_inches="tight")
plt.show()

## 4  Review Volume Over Time

Monthly aggregation smooths the noise while preserving the trend.  
HealthifyMe's dataset spans 2017–2026 but is much smaller (63K); MFP goes back to 2012 (653K).
Spikes in volume often coincide with major app updates or media coverage.

In [ ]:
monthly = (
    combined
    .assign(month=combined["review_date"].dt.to_period("M"))
    .groupby(["app_label", "month"])
    .size()
    .rename("count")
    .reset_index()
)
monthly["month_dt"] = monthly["month"].dt.to_timestamp()

print("Total monthly data points per app:")
print(monthly.groupby("app_label")["count"].describe().round(0).astype(int).to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

for ax, app in zip(axes, ["healthifyme", "myfitnesspal"]):
    sub = monthly[monthly["app_label"] == app]
    ax.fill_between(sub["month_dt"], sub["count"],
                    alpha=0.25, color=APP_COLORS[app])
    ax.plot(sub["month_dt"], sub["count"],
            color=APP_COLORS[app], linewidth=1.4)

    # 6-month rolling average
    rolled = sub.set_index("month_dt")["count"].rolling(6, min_periods=1).mean()
    ax.plot(rolled.index, rolled.values,
            color="black", linewidth=1.8, linestyle="--", alpha=0.6, label="6-mo rolling avg")

    label = app.replace("myfitnesspal", "MyFitnessPal").replace("healthifyme", "HealthifyMe")
    ax.set_title(f"{label} — Monthly Review Volume", fontsize=12, fontweight="bold")
    ax.set_ylabel("Reviews / month")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/04_volume_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Mean star rating over time (monthly)
monthly_rating = (
    combined
    .assign(month=combined["review_date"].dt.to_period("M"))
    .groupby(["app_label", "month"])["rating"]
    .mean()
    .reset_index()
)
monthly_rating["month_dt"] = monthly_rating["month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
for app in ["healthifyme", "myfitnesspal"]:
    sub = monthly_rating[monthly_rating["app_label"] == app]
    rolled = sub.set_index("month_dt")["rating"].rolling(6, min_periods=1).mean()
    label = app.replace("myfitnesspal", "MyFitnessPal").replace("healthifyme", "HealthifyMe")
    ax.plot(rolled.index, rolled.values,
            color=APP_COLORS[app], linewidth=2, label=label)

ax.set_title("Mean Star Rating Over Time (6-month rolling average)", fontsize=12, fontweight="bold")
ax.set_ylabel("Mean Rating")
ax.set_ylim(1, 5)
ax.axhline(3, color="grey", linestyle=":", linewidth=1)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_mean_rating_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

## 5  Top 15 Unigrams and Bigrams

Raw frequency after removing English stop words. These are the words users reach for most often —
useful for a quick sanity check that the text is domain-appropriate and as a baseline before
BERTopic surfaces latent topics.

We compute unigrams and bigrams separately per app so cross-app differences are visible.

In [ ]:
def top_ngrams(texts: pd.Series, n: int, top_k: int = 15) -> pd.DataFrame:
    """Return a DataFrame of the top_k n-grams by raw count."""
    vec = CountVectorizer(
        ngram_range=(n, n),
        stop_words="english",
        max_features=50_000,
        min_df=5,
        dtype=np.int32,
    )
    X = vec.fit_transform(texts.fillna(""))
    counts = np.asarray(X.sum(axis=0)).ravel()
    vocab  = vec.get_feature_names_out()
    return (
        pd.DataFrame({"ngram": vocab, "count": counts})
        .nlargest(top_k, "count")
        .reset_index(drop=True)
    )

print("Computing n-grams...")
hfy_uni = top_ngrams(hfy["review_description"], n=1)
hfy_bi  = top_ngrams(hfy["review_description"], n=2)
mfp_uni = top_ngrams(mfp["review_description"], n=1)
mfp_bi  = top_ngrams(mfp["review_description"], n=2)
print("Done.")

In [ ]:
# ── Tables ────────────────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    "HFY unigram":  hfy_uni["ngram"],  "HFY count":  hfy_uni["count"],
    "MFP unigram":  mfp_uni["ngram"],  "MFP count":  mfp_uni["count"],
})
print("Top 15 Unigrams:")
display(comparison)

comparison_bi = pd.DataFrame({
    "HFY bigram":  hfy_bi["ngram"],  "HFY count":  hfy_bi["count"],
    "MFP bigram":  mfp_bi["ngram"],  "MFP count":  mfp_bi["count"],
})
print("\nTop 15 Bigrams:")
display(comparison_bi)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

plot_data = [
    (axes[0][0], hfy_uni, "HealthifyMe — Top 15 Unigrams",  "healthifyme"),
    (axes[0][1], mfp_uni, "MyFitnessPal — Top 15 Unigrams", "myfitnesspal"),
    (axes[1][0], hfy_bi,  "HealthifyMe — Top 15 Bigrams",   "healthifyme"),
    (axes[1][1], mfp_bi,  "MyFitnessPal — Top 15 Bigrams",  "myfitnesspal"),
]

for ax, df, title, app in plot_data:
    df_sorted = df.sort_values("count")
    ax.barh(df_sorted["ngram"], df_sorted["count"],
            color=APP_COLORS[app], edgecolor="none", alpha=0.88)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Raw count")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.tick_params(axis="y", labelsize=9)

plt.suptitle("Top 15 Unigrams & Bigrams by Raw Frequency (stop words removed)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/06_ngrams.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary of Key EDA Findings

| Metric | HealthifyMe | MyFitnessPal |
|---|---|---|
| Cleaned rows | 63,147 | 653,650 |
| Date range | 2017 – 2026 | 2012 – 2026 |
| 5-star share | **65.3%** | **67.8%** |
| 1-star share | **13.7%** | **8.0%** |
| Median review length | 25 chars / 5 words | 65 chars / 12 words |
| Reviews < 20 chars (noise) | **44.9%** | 18.6% |
| Reviews ≥ 300 chars (long) | 4.3% | 5.4% |
| Developer response rate (overall) | **58.1%** | 10.7% |
| Dev response rate on 1-star reviews | **93.6%** | 62.4% |
| possible_mixed_sentiment | 964 (1.5%) | 19,252 (2.9%) |

**Notable signals for downstream modelling:**

1. **HFY has proportionally more 1-star reviews** (13.7% vs 8.0%) — suggests more acute dissatisfaction, likely tied to the paid coaching model. Both apps share dominant 5-star piles (65–68%) from satisfied casual users who never write long reviews.

2. **HFY reviews are far shorter** — median 25 chars / 5 words vs MFP's 65 chars / 12 words. Nearly half (44.9%) are under 20 characters ("good app", "👍"). BERTopic needs a minimum-length filter; these near-empty reviews will degrade topic quality if included.

3. **HFY's 58.1% dev-response rate is extraordinary**, especially on negative reviews (93.6% of 1-stars). HFY responds to almost every complaint; MFP largely ignores 4- and 5-star reviews. This asymmetry could bias sentiment comparisons: users who received a dev reply may have edited their rating upward.

4. **Bigrams already surface product narratives before any modelling:**
   - HFY top bigrams: `good app`, `nice app`, `indian food`, `diet plan` — feature-focused; local food database is a differentiator
   - MFP top bigrams: `easy use`, `great app`, `barcode scanner`, `weight loss`, `calorie intake` — utility-focused; `barcode scanner` and `bar code` in top 15 signals this feature drives both satisfaction and complaints when it breaks

5. **MFP's mixed-sentiment flag rate (2.9% = 19,252 rows) is nearly double HFY's (1.5% = 964 rows)** — consistent with the "used to be free, now paywalled" narrative. These rows are the sharpest test case for VADER vs transformer disagreement in notebook 04.